# RSNA Knee Abnormality Detection - Track B Specialist Training Pipeline
### Model: 2.5D Multi-View CNN with Multi-Instance Learning (MIL) Attention
- **Target Supervision**: `HYBRID_labels_scores.csv` (Verified 0.9024 Macro-AUC benchmark)
- **Validation Anchor**: 58 Gold Adjudicated Studies
- **Input**: Multi-View Knee MRI (Sagittal, Coronal, Axial)
- **Loss**: Multi-label Soft BCE with Logits

In [ ]:
import os
import gc
import time
import glob
import json
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Set seeds for exact reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training Device: {DEVICE}')

In [ ]:
FINDINGS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

def compute_macro_auc(y_true, y_score):
    y_true = np.asarray(y_true, dtype=bool)
    y_score = np.asarray(y_score, dtype=float)
    n_pos = np.sum(y_true)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    order = np.argsort(y_score)
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(len(y_score)) + 1
    unique_scores, inverse, counts = np.unique(y_score, return_inverse=True, return_counts=True)
    for i, count in enumerate(counts):
        if count > 1:
            ranks[inverse == i] = ranks[inverse == i].mean()
    return (np.sum(ranks[y_true]) - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)

print('AUC metric engine ready.')

In [ ]:
class MultiInstanceAttention(nn.Module):
    def __init__(self, in_features, hidden_dim=128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        attn_weights = self.attention(x)
        attn_weights = torch.softmax(attn_weights, dim=1)
        pooled = torch.sum(x * attn_weights, dim=1)
        return pooled, attn_weights

class KneeTrackBModel(nn.Module):
    def __init__(self, backbone_name='convnext_small', num_classes=12, pretrained=True):
        super().__init__()
        try:
            import timm
            self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
            in_features = self.backbone.num_features
        except:
            from torchvision.models import resnet34, ResNet34_Weights
            m = resnet34(weights=ResNet34_Weights.DEFAULT if pretrained else None)
            in_features = m.fc.in_features
            m.fc = nn.Identity()
            self.backbone = m

        self.pool = MultiInstanceAttention(in_features)
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        b, s, c, h, w = x.shape
        x = x.view(b * s, c, h, w)
        feats = self.backbone(x)
        feats = feats.view(b, s, -1)
        pooled, _ = self.pool(feats)
        logits = self.head(pooled)
        return logits

print('Track B model architecture initialized.')

In [ ]:
# Dataset Fold Creation and Soft Target Ingestion
DATA_DIR = '/kaggle/input/rsna-knee-abnormality-detection'
TRAIN_CSV = f'{DATA_DIR}/train.csv'
SERIES_CSV = f'{DATA_DIR}/train_series.csv'

# Check if running on Kaggle
if os.path.exists(TRAIN_CSV):
    train_df = pd.read_csv(TRAIN_CSV)
    series_df = pd.read_csv(SERIES_CSV)
    print(f'Loaded {len(train_df)} studies and {len(series_df)} series.')
else:
    print('Running in offline/local test mode.')

In [ ]:
# Training loop with early stopping on Gold Validation AUC
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(x)
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)

print('Training engine configured.')